---
<p align="center">
  <span style="color:Navy; font-size:200%; font-weight:bold; vertical-align:middle;">
        Dinámica del clima   
  </span>
</p>
<p align="center" style="line-height:1.2;">
  <span style="color:Blue; font-size:160%;">Unidad 2:Sistemas monzónicos</span><br/>
  <span style="color:Blue; font-size:140%;">Facultad de Ciencias  |  Semestre 2027-I</span><br/>
</p>

---

# **<font color="Navy">  Monzón Global </font>**
---
---

Este notebook esta basado en: https://github.com/adriantompkins/climate-book/blob/main/notebooks/part2_cda/CDA8_EOF_analysis.ipynb


### Terminología:

- FEO/EOF significa Funciones Ortogonales Empíricas (Empirical Orthogonal Functions)

- ACP/PCA significa Análisis de Componentes Principales (Principal Component Analysis)

- El análisis FEO/EOF y el ACOP/PCA son prácticamente lo mismo

- PCA es el término de uso más general, e implica simplemente convertir una matriz en conjuntos de variables linealmente no correlacionadas, llamadas componentes principales.

- El análisis EOF normalmente se refiere a encontrar los componentes principales que maximizan la varianza explicada (es decir, un subconjunto de todos los PC)

Supongamos que tenemos una matriz de datos $X$ que es función del tiempo y el espacio $X[t,s]$; por ahora, dejemos que las filas denoten distintos tiempos y las columnas distintas ubicaciones espaciales (aunque esto no es un requisito para la aplicación real).
En la práctica, a menudo aplicamos la técnica a datos 2D, es decir, función de x e y, esto es, $X=X[t,lon,lat]$, pero esto es esencialmente lo mismo; solo imagina tomar todos los puntos de lat/lon y alinearlos en un vector enorme y largo.

El objetivo del análisis EOF es descomponer $X$ en un conjunto de patrones espaciales, es decir, autovectores *ortogonales*, los $e_i$, que expliquen la mayor parte de la varianza de $X$. En otras palabras, queremos maximizar el parecido de $e_i$ (que tiene una dimensión espacial de $nlon*nlat$) con los datos, es decir, encontrar el vector $e_1$ que explique la mayor varianza.

La figura ofrece una visualización de lo que intentamos lograr. $X$ es una matriz de datos que consiste en la presión a nivel del mar (SLP, sea-level pressure) en mb, en $N$ puntos en el espacio y $M$ puntos en el tiempo.


![EOF](https://kls2177.github.io/Climate-and-Geophysical-Data-Analysis/_images/eofs.png)

El primer autovector $e_1$ muestra el patrón que explica la mayor parte de la varianza en $X$. Ten en cuenta que, al igual que los vectores unitarios en el espacio 2-D o 3-D, donde $e_1=(1,0)$ o $(-1,0)$ son autovectores igualmente válidos, el $e_1$ que describe la varianza de $X$ puede tener el aspecto del patrón mostrado en la Figura, o puede tener el aspecto de -1 veces ese patrón. Es decir, el análisis PCA produce un signo arbitrario en los patrones. Siguiendo estas reglas, el parecido de $e_1$ con los datos resulta ser:

$$(Xe_1)^2 = e_1^T X^T X e_1$$

Si $e_1$ es un vector unitario, entonces el objetivo de encontrar el $e_1$ con la máxima varianza es equivalente a la tarea de maximizar

$$ e_1^T C e_1$$

Si ese parecido máximo es igual a $\lambda_1$
entonces podemos escribir

$$ e_1^T C e_1=\lambda_1$$

lo cual reconocemos de inmediato como un problema de autovalores, ya que se puede escribir como

$$ e e_1^T C e_1= e\lambda_1$$

y por lo tanto

$$C e_1=\lambda_1 e_1$$

Así, $e_1$ debe ser un autovector de $C$ con su correspondiente autovalor $\lambda_1$. Por lo tanto, podemos encontrar $e_1$ "autoanalizando" C.

# Puntos clave:

- El 1er autovector corresponde al vector que explica la mayor parte de la varianza en $X$ y tiene el autovalor más grande

- El 2do autovector corresponde al vector que explica la segunda mayor varianza en $X$ y tiene el 2do autovalor más grande

- El análisis de autovalores de la matriz de covarianza transforma C en un sistema de coordenadas distinto, donde la "nueva" matriz es diagonal.

- En este nuevo espacio de coordenadas, toda la varianza está a lo largo de la diagonal, ya que los distintos vectores son ortogonales. Así, la fracción de varianza explicada por el $j$-ésimo autovector es el autovalor correspondiente dividido entre la suma de todos los autovalores, es decir

$\lambda_j / \sum \lambda_i$

Normalmente, las EOF se ordenan según su $\lambda_i$ correspondiente, de modo que la EOF 1 explica la mayor varianza (el $\lambda$ más grande) y la última EOF (típicamente la $n$-ésima EOF) tiene la menor.

¡Ninguna otra combinación lineal de $n$ predictores puede explicar una fracción de la varianza mayor que las primeras $n$ EOF/PC!

Recuerda también que los patrones (EOF) son ortogonales, es decir, su producto punto es cero. La serie temporal de componentes principales, Z, es equivalente a mapear X en el nuevo espacio vectorial generado por los autovectores. Podemos hacer esto realizando una transformación lineal de X.


![PC and EOF](https://kls2177.github.io/Climate-and-Geophysical-Data-Analysis/_images/pc_ts.png)

Como vimos en clase, las FEOs pueden ayudarnos a estudiar la variabilidad climática. 

En este caso usaremos esta metodología para estudia el **Monzón Global**. 

In [ ]:
#pip install netCDF4 h5netcdf

In [ ]:
import xarray as xr
import numpy as np 
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.animation import FuncAnimation, PillowWriter
import requests
import matplotlib.dates as mdates

In [ ]:
#Subimos los datos
url = "https://github.com/Dinamica-del-Clima-2027-1/Sistemas_monzonicos/raw/refs/heads/main/data/era5_precipitation_monthly_climatology_1deg.nc"
archivo_local = "era5_precipitation_monthly_climatology_1deg.nc"

# Descargamos el archivo
response = requests.get(url)
response.raise_for_status()  # lanza error si la descarga falló (ej. 404)

with open(archivo_local, "wb") as f:
    f.write(response.content)

# Ahora sí lo abrimos localmente
ds = xr.open_dataset(archivo_local)
ds

In [ ]:
ds['tp']

In [ ]:
#Convertimos la precipitacion a milimetros
precip = ds["tp"]*1000

#Extraemos las latitudes y longitutes
lat = ds["lat"]
lon = ds["lon"]

## <font color ="DarkBlue"> Preprocesamiento de datos </font>

<font color = "blue"> Remosion del promedio </font>

Es habitual eliminar la media de la dimensión de muestreo, que normalmente es la dimensión temporal. Esto hará que la serie temporal de componentes principales tenga una media de cero. También podrías querer eliminar el ciclo estacional o diurno si no es de tu interés.

Es menos común eliminar la media espacial. Por ejemplo, supongamos que estás interesado en los gradientes de temperatura, no en el campo de temperatura completo, o que te interesa la componente "ondulada" del campo, en cuyo caso podrías eliminar la media zonal.

In [ ]:
# Calculo del promedio temporal para cada punto de la malla (lon, lat)
precip_mean = precip.mean(dim="valid_time")
# Calculo de la desviacion estandar temporal para cada punto de la malla
precip_std = precip.std(dim="valid_time")

# Calculo de las anomalias de precipitacion
precip_anom_real = (precip - precip_mean)


<font color = "Blue"> Estandarización de los datos</font>

A veces, es conveniente eliminar la amplitud de la variabilidad de los datos antes de realizar el análisis EOF. Por ejemplo, podrías querer hacer esto si tus datos están formados por una combinación de parámetros con distintas unidades, y no quieres que el parámetro con las unidades más grandes domine la varianza.

Para hacer esto, divides tus datos de anomalía también entre su desviación estándar (ahora tus datos están en unidades de $\sigma$). Para el ejemplo de precipitación, no estandarizaremos los datos, ya que estamos interesados en capturar la magnitud espacial de la variabilidad. 

In [ ]:
# Estandarizacion de los datos

#precip_anom = precip_anom_real / precip_std
precip_anom = precip_anom_real

<font color = "Blue"> Ponderacion de los datos de la malla </font>

Si los datos están en una malla (gridded), es necesario ponderarlos según el área de la caja de la malla (grid box). Por ejemplo, si los datos están en una cuadrícula rectangular sobre una esfera (como en nuestro ejemplo), el tamaño de las cajas de la cuadrícula disminuye a medida que te desplazas del ecuador hacia los polos.

Si estás calculando las EOF usando la matriz de covarianza y estás analizando la matriz de covarianza temporal (como aquí, y como suele ser el caso), entonces necesitamos ponderar por $\sqrt{(cos(lat))}$, porque la matriz de covarianza es $XX^T$.

Ten en cuenta que las EOF resultantes se verán poco físicas y tendrán valores poco físicos; esto se solucionará cuando veamos cómo presentar las EOF (ver más abajo).

In [ ]:
# Se convierten las latitudes a radianes
lat_rad = (ds['lat'] * np.pi / 180.0)

#Se calcula el coseno de los pesos
cos_wgt = np.sqrt(np.cos(lat_rad))

#Se aplica a toda la malla
precip_mean = precip_anom*cos_wgt

<font color = "Blue"> De un arreglo 3D a uno 2D </font>

El siguiente paso es convertir un arreglo 3D a una matriz 2D, de modo que podamos realizar operaciones con matrices.

In [ ]:
# Se redimensionan los datos a (ntime, nlon*nlat)
# Extraccion de las dimensiones
ntime, nlat, nlon = precip_anom.sizes['valid_time'], precip_anom.sizes['lat'], precip_anom.sizes['lon']
precip_reshape = precip_anom.values.reshape(ntime,nlon * nlat)
precip_anom = xr.DataArray(
            precip_reshape,
            dims=['valid_time', 'space'],
            coords={'valid_time': precip_anom.coords['valid_time']} 
        )

precip_anom.shape

<font color = "Blue"> Cálculo de la matriz de covarianza </font>

Nuestros datos estan listos para calcular la matriz de covarianza. Recuera que vamos a calcular la matriz de covarianza temporal debido a los recursos computacionales que se requieren respecto de las matrices de dispersion espaciales.

In [ ]:
# calcular de la matriz de covarianza
C = np.cov(precip_anom)
print(C.shape)

<font color = "Blue"> Cálculo de los eigenvectores </font>

El siguiente paso es el analisis de los eigenvectores. Nota que, como estamos analizando la matriz de covarianza temporal, los eigenvectores corresponden a las componentes principales de las series de tiempo y calcularemos los patrones espaciales de las FEOs usando una transformacion lineal. 

In [ ]:
# LAM=Lambda eigenvalues, 
# Nos dice la varianza explicada
# E=eigenvectores, Las componentes principales (PCs)...

LAM, E = np.linalg.eig(C)

In [ ]:
E.shape

Ahora podemos extraer ese patrón proyectando la matriz de datos ponderada sobre la serie temporal de PC, ya que esta es la que se usó para calcular la matriz de covarianza.

Podemos hacer esto tomando el producto punto entre el campo original y el pc1:

``np.dot(precip_anom.T,pc1)``

Así, cualquier variable que covaríe con el PC terminará mostrando una señal en el campo resultante del producto punto.

**NOTA** Podríamos aplicar este producto punto a cualquier otro campo para observar cómo covaría con el índice, pero antes es mejor normalizar la serie del PC, a lo cual volveremos más adelante

In [ ]:
# Numero de FEO que queremos estudiar.
neof=5 

# Listas vacias  
eof_da=[]
pc_da=[]

for ieof in range(neof):
    # Calculo de la primer componente principal
    # Las FEOs se calculan proyectando la matriz de datos en las series de 
    #tiempo de las componente principales
    EOF  = np.reshape(np.dot(precip_anom.T,E[:,ieof]), (nlat,nlon)) 

# Se crea un DataArray para el patron FEO con coordenadas lat/lon
    eof_da.append(xr.DataArray(
        EOF,
        coords={'lat': precip.coords['lat'], 'lon': precip.coords['lon']},
        dims=['lat', 'lon']))
    pc_da.append(xr.DataArray(E[:,ieof], coords={'valid_time': precip.coords['valid_time']}, dims=['valid_time']))


Graficamos la primer FEO y la serie de tiempo de la CP.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(13, 8)) 

# En el primer panel se grafica el patron FEO
ax[0] = plt.subplot(2, 1, 1, projection=ccrs.PlateCarree())
eof_da[0].plot(ax=ax[0], cmap='RdBu', transform=ccrs.PlateCarree(),vmin=-40, vmax=40)
ax[0].set_title('Patron de la primer FEO')
ax[0].coastlines()
ax[0].add_feature(cfeature.BORDERS)
ax[0].spines['bottom'].set_visible(False)
ax[0].spines['top'].set_visible(False)
ax[0].spines['left'].set_visible(False)
ax[0].spines['right'].set_visible(False)

# Se grafica la serie de tiempo del primer componente principal
pc_da[0].plot(ax=ax[1])
ax[1].set_title('Serie de tiempo de la primer Componente Principal')
ax[1].set_xlabel('Tiempo')
ax[1].set_ylabel('Amplitud')
ax[1].grid(True)

plt.tight_layout()
plt.show()

### <font color = "Blue"> Presentación de EOF y PC </font>



Las EOF y las series temporales de las PC resultantes del análisis de autovalores (eigenanálisis) generalmente no se expresan en unidades físicas. Recordemos que hemos aplicado una ponderación por coseno a los datos y que los autovectores son vectores unitarios. Por tanto, es necesario realizar algunos pasos de posprocesamiento para presentar las EOF y las PC de manera significativa.

Por lo general, normalizamos la serie temporal de la PC dividiéndola por su desviación estándar; así, basta con estandarizarla para crear un índice.

A continuación, proyectamos esta PC normalizada sobre los datos de anomalía originales y dividimos por la longitud de la serie para obtener la estructura de la EOF 1 en las unidades originales (es decir, las de los datos sin ponderar).

Cabe señalar que también es posible realizar proyecciones sobre otras variables. Por ejemplo, en el caso del índice ENSO 3.4, podemos proyectar la temperatura o el viento zonal sobre la PC correspondiente para observar cómo varían dichas variables en relación con el fenómeno ENSO.

In [ ]:
pc_ts_std=[]
for pc in pc_da: 
   pc_ts_std.append((pc - np.mean(pc))/np.std(pc))

In [ ]:
# Convertir las FEO a unidades fisicas 

# Primero se redimensionan los dado de 3D a 2D
precip_reshape = precip_anom_real.values.reshape(ntime,nlon * nlat)

In [ ]:
EOF_phys=[]

for pc in pc_ts_std: 
    EOF_phys1 = np.dot(pc.T,precip_reshape*(1/ntime))
    #redimensionar de nuevo para graficar
    EOF_phys1 = np.reshape(EOF_phys1,(nlat,nlon))

    EOF_phys.append(xr.DataArray(
        EOF_phys1,
        coords={'lat': precip.coords['lat'], 'lon': precip.coords['lon']},
        dims=['lat', 'lon'] ))

In [ ]:
###################
peof=1  # peof=1 es la primer FEP

if peof>neof:
    print ("Bad plot eof, we only calculated neof=",neof, " resetting to EOF=1")
    peof=1

########



fig = plt.figure(figsize=(13, 16))

# ---- Primer gráfico: patrón espacial de la EOF ----
ax0 = fig.add_subplot(2, 1, 1, projection=ccrs.PlateCarree())

im = EOF_phys[peof-1].plot(
    ax=ax0, cmap='RdBu_r', transform=ccrs.PlateCarree(),
    vmin=-5, vmax=5, add_colorbar=False
)
ax0.set_title('Patrón de EOF' + str(peof))
ax0.coastlines()
ax0.add_feature(cfeature.BORDERS)
#ax0.tick_params(axis='x', labelsize=6, labelrotation=90)  # para que no se amontonen las etiquetas
ax0.xaxis.set_visible(True)
ax0.yaxis.set_visible(True)

gl = ax0.gridlines(crs=ccrs.PlateCarree(), draw_labels=False,
                    linewidth=1, color='gray', alpha=0.5, linestyle='--')

for spine in ax0.spines.values():
    spine.set_visible(False)

# ---- Segundo gráfico: serie temporal del PC ----
ax1 = fig.add_subplot(2, 1, 2)
pc_ts_std[peof-1].plot(ax=ax1)
ax1.set_title('PC' + str(peof) + ' Time Series')
ax1.set_xlabel('Time')
ax1.set_ylabel('Amplitude')
ax1.grid(True)

# Un tick por cada año, mostrando solo el año
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')

fig.colorbar(im, ax=[ax0, ax1], orientation='horizontal', pad=0.1, label='mm/day')

plt.show()

<font color = "Blue"> ¿Cuántas FEOs/CPs son importantes? </font >

El análisis de autovalores de la matriz de covarianza siempre te dará una respuesta. La ortogonalidad de los autovectores/EOF impone una restricción sobre su patrón. Las EOF de orden superior a menudo intentan representar el ruido, manteniéndose a la vez ortogonales a las demás EOF, y en consecuencia, estos patrones pueden convertirse en construcciones estadísticas espurias en lugar de modos de variabilidad físicamente significativos.


In [ ]:
# set up plot
plt.figure(figsize=(10,6))

# plot fraction of variance explained by first 10 eigenvectors using the eigenvalues
plt.plot(np.arange(1,np.size(LAM[:10])+1.),LAM[:10] / LAM.sum(),'.-',color='gray',linewidth=2)
plt.xlim(0.5, 10.5)

# define N*
Nstar = len(pc_ts_std)

# compute error bars using the North et al. "rule of thumb"
eb = LAM[:10] / LAM.sum()*np.sqrt(2./float(Nstar))
plt.errorbar(np.arange(1,np.size(LAM[:10])+1.),LAM[:10] / LAM.sum(),yerr = eb/float(2), xerr = None, linewidth = 1, color = 'gray')

# add labels, title, etc.
plt.title('Fracción de Varianza Explicada',fontsize=16)
plt.xlabel('FEOs')
plt.ylabel('Varianza Explicada')
plt.show()

---
### **<font color="DodgerBlue">Ejercicio: </font>**

Muestren el patrón espacial de la segunda y tercera FEO y las series de tiempo de sus respectivas CPs. 

¿Qué diferencias encuentran comparando con la primera FEO  y CP? 
¿Qué interpretación Física tienen estos patrones?


---